In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("MessyDataset").getOrCreate()

# Sample messy data
data = [
    (1, "Alice", 29, "Sales", 5000),
    (2, "BOB", None, "IT", 6000),
    (3, "Charlie", 28, None, 4500),
    (4, "David", 40, "IT", None),
    (5, None, 35, "Sales", 6500),
    (6, "eve", None, "HR", 4000),
    (7, "Frank", 50, "IT", 8000),
    (8, "Grace", 30, "HR", None),
    (9, "HENRY", 27, "Sales", 5200),
    (10, None, None, None, None)
]

# Define schema
columns = ["id", "name", "age", "department", "salary"]

# Create DataFrame
df = spark.createDataFrame(data, schema=columns)

# Show the messy DataFrame
df.show(truncate=False)

+---+-------+----+----------+------+
|id |name   |age |department|salary|
+---+-------+----+----------+------+
|1  |Alice  |29  |Sales     |5000  |
|2  |BOB    |NULL|IT        |6000  |
|3  |Charlie|28  |NULL      |4500  |
|4  |David  |40  |IT        |NULL  |
|5  |NULL   |35  |Sales     |6500  |
|6  |eve    |NULL|HR        |4000  |
|7  |Frank  |50  |IT        |8000  |
|8  |Grace  |30  |HR        |NULL  |
|9  |HENRY  |27  |Sales     |5200  |
|10 |NULL   |NULL|NULL      |NULL  |
+---+-------+----+----------+------+



In [0]:
df.fillna({"age": 0, "name": "Unknown", "department": "Unknown", "salary": 0}).show()

+---+-------+---+----------+------+
| id|   name|age|department|salary|
+---+-------+---+----------+------+
|  1|  Alice| 29|     Sales|  5000|
|  2|    BOB|  0|        IT|  6000|
|  3|Charlie| 28|   Unknown|  4500|
|  4|  David| 40|        IT|     0|
|  5|Unknown| 35|     Sales|  6500|
|  6|    eve|  0|        HR|  4000|
|  7|  Frank| 50|        IT|  8000|
|  8|  Grace| 30|        HR|     0|
|  9|  HENRY| 27|     Sales|  5200|
| 10|Unknown|  0|   Unknown|     0|
+---+-------+---+----------+------+



In [0]:
df.withColumn("name", lower(col("name"))).show()

+---+-------+----+----------+------+
| id|   name| age|department|salary|
+---+-------+----+----------+------+
|  1|  alice|  29|     Sales|  5000|
|  2|    bob|NULL|        IT|  6000|
|  3|charlie|  28|      NULL|  4500|
|  4|  david|  40|        IT|  NULL|
|  5|   NULL|  35|     Sales|  6500|
|  6|    eve|NULL|        HR|  4000|
|  7|  frank|  50|        IT|  8000|
|  8|  grace|  30|        HR|  NULL|
|  9|  henry|  27|     Sales|  5200|
| 10|   NULL|NULL|      NULL|  NULL|
+---+-------+----+----------+------+



In [0]:
df.dropna(subset=["name", "department"]).show()

+---+-----+----+----------+------+
| id| name| age|department|salary|
+---+-----+----+----------+------+
|  1|Alice|  29|     Sales|  5000|
|  2|  BOB|NULL|        IT|  6000|
|  4|David|  40|        IT|  NULL|
|  6|  eve|NULL|        HR|  4000|
|  7|Frank|  50|        IT|  8000|
|  8|Grace|  30|        HR|  NULL|
|  9|HENRY|  27|     Sales|  5200|
+---+-----+----+----------+------+



In [0]:
avg_salary = df.select(avg("salary")).first()[0]
df.fillna({"salary": avg_salary}).show()

+---+-------+----+----------+------+
| id|   name| age|department|salary|
+---+-------+----+----------+------+
|  1|  Alice|  29|     Sales|  5000|
|  2|    BOB|NULL|        IT|  6000|
|  3|Charlie|  28|      NULL|  4500|
|  4|  David|  40|        IT|  5600|
|  5|   NULL|  35|     Sales|  6500|
|  6|    eve|NULL|        HR|  4000|
|  7|  Frank|  50|        IT|  8000|
|  8|  Grace|  30|        HR|  5600|
|  9|  HENRY|  27|     Sales|  5200|
| 10|   NULL|NULL|      NULL|  5600|
+---+-------+----+----------+------+



In [0]:
df.filter((col("department") == "IT") & (col("salary") > 6000)).show()

+---+-----+---+----------+------+
| id| name|age|department|salary|
+---+-----+---+----------+------+
|  7|Frank| 50|        IT|  8000|
+---+-----+---+----------+------+



In [0]:
df.withColumn("age_group", when(col("age") < 30, "Young")
                             .when(col("age") <= 40, "Mid")
                             .otherwise("Senior")).show()

+---+-------+----+----------+------+---------+
| id|   name| age|department|salary|age_group|
+---+-------+----+----------+------+---------+
|  1|  Alice|  29|     Sales|  5000|    Young|
|  2|    BOB|NULL|        IT|  6000|   Senior|
|  3|Charlie|  28|      NULL|  4500|    Young|
|  4|  David|  40|        IT|  NULL|      Mid|
|  5|   NULL|  35|     Sales|  6500|      Mid|
|  6|    eve|NULL|        HR|  4000|   Senior|
|  7|  Frank|  50|        IT|  8000|   Senior|
|  8|  Grace|  30|        HR|  NULL|      Mid|
|  9|  HENRY|  27|     Sales|  5200|    Young|
| 10|   NULL|NULL|      NULL|  NULL|   Senior|
+---+-------+----+----------+------+---------+

